In [1]:
!pip install Pillow imagehash matplotlib numpy pandas

In [2]:
import hashlib
from PIL import Image, ImageDraw
import imagehash
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. Simulate generating target/reference "hashed" fingerprints (representing known flagged media signatures)
print("[INFO] Initializing Thorn-inspired perceptual hashing reference database...")
np.random.seed(42)

# Create synthetic sample reference images to act as our "database"
reference_hashes = {}
for i in range(1, 6):
    # Generate a dummy image with random noise/shapes
    img = Image.new('RGB', (200, 200), color=(np.random.randint(0, 255), np.random.randint(0, 255), np.random.randint(0, 255)))
    d = ImageDraw.Draw(img)
    d.rectangle([i*10, i*10, 100+i*5, 100+i*5], fill=(255, 255, 255))

    # Compute Perceptual Hash (phash) - robust against resizing, compression, and minor edits
    p_hash = imagehash.phash(img)
    reference_hashes[f"flagged_hash_ref_{i}"] = p_hash

print(f"[INFO] Loaded {len(reference_hashes)} reference fingerprints into secure memory.")

# 2. Simulate incoming media stream (e.g., user-uploaded images or chat attachments)
# We will create an incoming image that is a slightly modified version of reference #3 (e.g., cropped or compressed)
target_img = Image.new('RGB', (200, 200), color=(np.random.randint(0, 255), np.random.randint(0, 255), np.random.randint(0, 255)))
d = ImageDraw.Draw(target_img)
d.rectangle([3*10, 3*10, 100+3*5, 100+3*5], fill=(255, 255, 255)) # matches ref_3 shape
target_hash = imagehash.phash(target_img)

# 3. Execute Distance Matching (Hamming Distance Search)
MATCH_THRESHOLD = 5  # Lower distance means higher visual similarity
match_results = []

for ref_id, ref_hash in reference_hashes.items():
    hamming_distance = target_hash - ref_hash # Calculates bit difference
    is_match = hamming_distance <= MATCH_THRESHOLD
    match_results.append({
        'reference_id': ref_id,
        'hamming_distance': hamming_distance,
        'flagged_actionable': is_match
    })

results_df = pd.DataFrame(match_results)
print("\n--- Media Fingerprint Match Analysis ---")
print(results_df.to_string(index=False))

# 4. Automated Moderation Decision
matched_hits = results_df[results_df['flagged_actionable'] == True]
if not matched_hits.empty:
    print(f"\n[ALERT] Incoming media matched flagged hash signatures! Action: Instant blur/quarantine initiated.")
else:
    print(f"\n[CLEAN] Incoming media cleared safety thresholds.")

[INFO] Initializing Thorn-inspired perceptual hashing reference database...
[INFO] Loaded 5 reference fingerprints into secure memory.

--- Media Fingerprint Match Analysis ---
      reference_id  hamming_distance  flagged_actionable
flagged_hash_ref_1                24               False
flagged_hash_ref_2                12               False
flagged_hash_ref_3                 0                True
flagged_hash_ref_4                14               False
flagged_hash_ref_5                32               False

[ALERT] Incoming media matched flagged hash signatures! Action: Instant blur/quarantine initiated.


In [3]:
import datetime

# --- ADVANCED STEP 4: BATCH INGESTION & AUTOMATED INCIDENT TRIAGE PIPELINE ---

print("[INFO] Initializing batch moderation scan across incoming media queue...")

# 1. Simulate a batch queue of incoming user media uploads (varying modifications)
batch_queue = []
np.random.seed(101)

for idx in range(1, 16): # 16 incoming uploads
    # Create varying image distortions (simulating compression, cropping, or edits)
    img_batch = Image.new('RGB', (200, 200), color=(np.random.randint(0, 255), np.random.randint(0, 255), np.random.randint(0, 255)))
    d_batch = ImageDraw.Draw(img_batch)
    # Inject shapes mimicking different risk tiers
    shape_size = np.random.choice([80, 100, 120])
    d_batch.rectangle([10, 10, shape_size, shape_size], fill=(np.random.randint(0, 255), np.random.randint(0, 255), np.random.randint(0, 255)))

    batch_queue.append({
        'upload_id': f"UPLOAD_90{idx}",
        'image_obj': img_batch,
        'timestamp': datetime.datetime.now(datetime.timezone.utc).isoformat()
    })

# 2. Process Batch & Execute Multi-Tier Threat Evaluation
moderation_audit_records = []
STRICT_MATCH_THRESHOLD = 3   # High confidence match
REVIEW_MATCH_THRESHOLD = 7   # Moderate similarity / suspicious modification

for item in batch_queue:
    current_hash = imagehash.phash(item['image_obj'])

    closest_match_id = None
    min_distance = 999

    # Compare against all known reference signatures in our secure database
    for ref_id, ref_hash in reference_hashes.items():
        dist = current_hash - ref_hash
        if dist < min_distance:
            min_distance = dist
            closest_match_id = ref_id

    # Assign automated safety disposition based on Hamming distance tiers
    if min_distance <= STRICT_MATCH_THRESHOLD:
        status_disposition = "CRITICAL_QUARANTINE"
        action_taken = "Instant Blur & Account Flagged for Review"
    elif min_distance <= REVIEW_MATCH_THRESHOLD:
        status_disposition = "REVIEW_QUEUE"
        action_taken = "Forwarded to Trust & Safety Human Reviewers"
    else:
        status_disposition = "CLEAN_PASSED"
        action_taken = "Delivered Normally"

    moderation_audit_records.append({
        'upload_id': item['upload_id'],
        'timestamp': item['timestamp'],
        'closest_reference_match': closest_match_id,
        'hamming_distance': int(min_distance),
        'disposition': status_disposition,
        'action_executed': action_taken
    })

# 3. Compile Audit DataFrame & Display Metrics
audit_df = pd.DataFrame(moderation_audit_records)

print(f"\n[SUMMARY] Batch moderation complete. Processed {len(audit_df)} incoming media payloads.")
print("\n--- Triage Disposition Breakdown ---")
print(audit_df['disposition'].value_counts().to_string())

print("\n--- Sample Audit Log Entries ---")
print(audit_df[['upload_id', 'hamming_distance', 'disposition']].head(6).to_string(index=False))

# 4. Export Immutable Audit Report for Compliance/Investigators
audit_filename = "moderation_incident_audit.csv"
audit_df.to_csv(audit_filename, index=False)
print(f"\n[INFO] Immutable compliance audit log successfully compiled and exported to '{audit_filename}'.")

[INFO] Initializing batch moderation scan across incoming media queue...

[SUMMARY] Batch moderation complete. Processed 15 incoming media payloads.

--- Triage Disposition Breakdown ---
disposition
CLEAN_PASSED           14
CRITICAL_QUARANTINE     1

--- Sample Audit Log Entries ---
 upload_id  hamming_distance  disposition
UPLOAD_901                14 CLEAN_PASSED
UPLOAD_902                32 CLEAN_PASSED
UPLOAD_903                30 CLEAN_PASSED
UPLOAD_904                32 CLEAN_PASSED
UPLOAD_905                28 CLEAN_PASSED
UPLOAD_906                32 CLEAN_PASSED

[INFO] Immutable compliance audit log successfully compiled and exported to 'moderation_incident_audit.csv'.
